## Import Library


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import time

# Extract data

In [3]:
from pathlib import Path


def _resolve_path(file_path: str) -> Path:
    path = Path(file_path)
    if path.is_file():
        return path
    root = Path.cwd().parent if Path.cwd().name == "etl" else Path.cwd()
    resolved = root / file_path
    if resolved.is_file():
        return resolved
    raise FileNotFoundError(f"File not found: {file_path!r} (tried {resolved})")


def extract_data(file_path: str, file_type: str) -> pd.DataFrame:
    """Load banking (or other tabular) data from disk into a DataFrame."""
    file_type = file_type.lower().strip()
    path = _resolve_path(file_path)

    match file_type:
        case "csv":
            return pd.read_csv(path)
        case "excel" | "xlsx" | "xls":
            return pd.read_excel(path)
        case "json":
            return pd.read_json(path)
        case _:
            raise ValueError(
                f"Unsupported file_type={file_type!r}. Use csv, excel, or json."
            )

# data frame
df = extract_data("data/bank_marketing.csv", "csv")

## 01- understand the dataset

In [4]:
# understand the dataset
print('#########################header#########################')
print(df.head(5))
print('#########################tail#########################')
print(df.tail(5))
print('#########################info#########################')
print(df.info())
print('#########################describe#########################')
print(df.describe())
print('#########################shape#########################')
print(df.shape)
print('#########################columns#########################')
print(df.columns)
print('#########################dtypes#########################')
print(df.dtypes)
print("######################count columns######################")
print(df.count())

#########################header#########################
   client_id  age        job  marital    education credit_default mortgage  \
0          0   56  housemaid  married     basic.4y             no       no   
1          1   57   services  married  high.school        unknown       no   
2          2   37   services  married  high.school             no      yes   
3          3   40     admin.  married     basic.6y             no       no   
4          4   56   services  married  high.school             no       no   

  month  day  contact_duration  number_contacts  previous_campaign_contacts  \
0   may   13               261                1                           0   
1   may   19               149                1                           0   
2   may   23               226                1                           0   
3   may   27               151                1                           0   
4   may    3               307                1                           0   


## Cleaning Data

In [5]:
# clean data
# check for missing values
print('#########################missing values#########################')
print(df.isnull().sum())

# check for duplicate values
print('#########################duplicate values#########################')
print(df.duplicated().sum())

# check for outliers
print('#########################outliers#########################')
print(df.describe())


#########################missing values#########################
client_id                     0
age                           0
job                           0
marital                       0
education                     0
credit_default                0
mortgage                      0
month                         0
day                           0
contact_duration              0
number_contacts               0
previous_campaign_contacts    0
previous_outcome              0
cons_price_idx                0
euribor_three_months          0
campaign_outcome              0
dtype: int64
#########################duplicate values#########################
0
#########################outliers#########################
         client_id          age           day  contact_duration  \
count  41188.00000  41188.00000  41188.000000      41188.000000   
mean   20593.50000     40.02406     15.730334        258.285010   
std    11890.09578     10.42125      8.889867        259.279249   
min        0.0

In [6]:
VALUE_COUNT_COLUMNS = [
    "client_id",
    "age",
    "job",
    "marital",
    "education",
    "credit_default",
    "mortgage",
    "month",
    "day",
    "contact_duration",
    "number_contacts",
    "previous_campaign_contacts",
    "previous_outcome",
    "cons_price_idx",
    "euribor_three_months",
    "campaign_outcome",
]


## Counting Data

In [7]:
def count_data(df: pd.DataFrame, columns: list[str], dropna: bool = False) -> pd.DataFrame:
    for col in columns:
        print(f"#########################{col} ({df[col].dtype})#########################")
        counts = df[col].value_counts(dropna=dropna)
        if counts.shape[0] > 30:
            print(f"unique values: {counts.shape[0]}")
            print(counts.head(30))

## Print Including missing values

In [8]:
countingWithMissingValues = count_data(df, VALUE_COUNT_COLUMNS, dropna=False)

#########################client_id (int64)#########################
unique values: 41188
client_id
0     1
1     1
2     1
3     1
4     1
5     1
6     1
7     1
8     1
9     1
10    1
11    1
12    1
13    1
14    1
15    1
16    1
17    1
18    1
19    1
20    1
21    1
22    1
23    1
24    1
25    1
26    1
27    1
28    1
29    1
Name: count, dtype: int64
#########################age (int64)#########################
unique values: 78
age
31    1947
32    1846
33    1833
36    1780
35    1759
34    1745
30    1714
37    1475
29    1453
39    1432
38    1407
41    1278
40    1161
42    1142
45    1103
43    1055
46    1030
44    1011
28    1001
48     979
47     928
50     875
27     851
49     839
52     779
51     754
53     733
56     704
26     698
54     684
Name: count, dtype: int64
#########################job (str)#########################
#########################marital (str)#########################
#########################education (str)#########################
#####

# Print without missing values

In [9]:
countingWithoutMissingValues = count_data(df, VALUE_COUNT_COLUMNS, dropna=True)


#########################client_id (int64)#########################
unique values: 41188
client_id
0     1
1     1
2     1
3     1
4     1
5     1
6     1
7     1
8     1
9     1
10    1
11    1
12    1
13    1
14    1
15    1
16    1
17    1
18    1
19    1
20    1
21    1
22    1
23    1
24    1
25    1
26    1
27    1
28    1
29    1
Name: count, dtype: int64
#########################age (int64)#########################
unique values: 78
age
31    1947
32    1846
33    1833
36    1780
35    1759
34    1745
30    1714
37    1475
29    1453
39    1432
38    1407
41    1278
40    1161
42    1142
45    1103
43    1055
46    1030
44    1011
28    1001
48     979
47     928
50     875
27     851
49     839
52     779
51     754
53     733
56     704
26     698
54     684
Name: count, dtype: int64
#########################job (str)#########################
#########################marital (str)#########################
#########################education (str)#########################
#####

In [10]:
print(df['mortgage'].head(100))

0      no
1      no
2     yes
3      no
4      no
     ... 
95    yes
96     no
97     no
98     no
99     no
Name: mortgage, Length: 100, dtype: str


# Split Cv and Save

In [11]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "etl" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(df.dtypes)
# Cleaning
client = df[
    [
        "client_id",
        "age",
        "job",
        "marital",
        "education",
        "credit_default",
        "mortgage",
    ]
].copy()


client_id                       int64
age                             int64
job                               str
marital                           str
education                         str
credit_default                    str
mortgage                          str
month                             str
day                             int64
contact_duration                int64
number_contacts                 int64
previous_campaign_contacts      int64
previous_outcome                  str
cons_price_idx                float64
euribor_three_months          float64
campaign_outcome                  str
dtype: object


 # Save Csv && Cleaning

In [12]:

client["job"] = client["job"].str.replace(".", "_", regex=False)
client["education"] = client["education"].str.replace(".", "_", regex=False).replace("unknown", np.nan)

# make credit_default as boolean data type
client['credit_default'] =  (
    client['credit_default'] == "yes"
).astype(int)

# mortage
client["mortgage"] = (
    client["mortgage"] == "yes"
).astype(int)



## Ensure Cleaning data

In [13]:
listOfClient = [
    "job",
    "education",
    "credit_default",
    "mortgage"
]

for col in listOfClient:
    print(f"\n{col}:")
    print(client[col].value_counts(dropna=False))

# count_data(client, listOfClient, False)




job:
job
admin_           10422
blue-collar       9254
technician        6743
services          3969
management        2924
retired           1720
entrepreneur      1456
self-employed     1421
housemaid         1060
unemployed        1014
student            875
unknown            330
Name: count, dtype: int64

education:
education
university_degree      12168
high_school             9515
basic_9y                6045
professional_course     5243
basic_4y                4176
basic_6y                2292
NaN                     1731
illiterate                18
Name: count, dtype: int64

credit_default:
credit_default
0    41185
1        3
Name: count, dtype: int64

mortgage:
mortgage
1    21576
0    19612
Name: count, dtype: int64


In [14]:
client.to_csv(PROCESSED_DIR / "client.csv", index=False)
print(f"Saved {PROCESSED_DIR / 'client.csv'}")


Saved /home/taha/projects/AI/data_engineer/projects/clean_banking_data/data/processed/client.csv


## Split campaign

In [27]:
# print(df.info())

campaign = df[
    [
        "client_id",
        "contact_duration",
        "number_contacts",
        "previous_campaign_contacts",
        "previous_outcome",
        "campaign_outcome",
        "month",
        "day",
        # "last_contact_date"
    ]
].copy()
print(campaign.info())


<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   client_id                   41188 non-null  int64
 1   contact_duration            41188 non-null  int64
 2   number_contacts             41188 non-null  int64
 3   previous_campaign_contacts  41188 non-null  int64
 4   previous_outcome            41188 non-null  str  
 5   campaign_outcome            41188 non-null  str  
 6   month                       41188 non-null  str  
 7   day                         41188 non-null  int64
dtypes: int64(5), str(3)
memory usage: 2.5 MB
None


#  Clean campaign

In [30]:
listOfcompaign=[
        "client_id",
        "contact_duration",
        "number_contacts",
        "previous_campaign_contacts",
        "previous_outcome",
        "campaign_outcome",
        "last_contact_date"
    ]


In [33]:
campaign["previous_outcome"]=(
    campaign["previous_outcome"] == "success"
).astype(int)

campaign["campaign_outcome"]=(
    campaign["campaign_outcome"] == "yes"
).astype(int)

campaign["year"] = 2022

campaign["month"] = campaign["month"].str.title()

campaign["last_contact_date"] = pd.to_datetime(
    campaign["year"].astype(str)
    + "-"
    + campaign["month"]
    + "-"
    + campaign["day"].astype(str),
    format="%Y-%b-%d"
)

In [ ]:

for col in listOfcompaign:
    print(f"\n{col}:")
    print(campaign[col].value_counts(dropna=False))

    


client_id:
client_id
0        1
1        1
2        1
3        1
4        1
        ..
41183    1
41184    1
41185    1
41186    1
41187    1
Name: count, Length: 41188, dtype: int64

contact_duration:
contact_duration
85      170
90      170
136     168
73      167
124     164
       ... 
1640      1
1616      1
1246      1
1556      1
1868      1
Name: count, Length: 1544, dtype: int64

number_contacts:
number_contacts
1     17642
2     10570
3      5341
4      2651
5      1599
6       979
7       629
8       400
9       283
10      225
11      177
12      125
13       92
14       69
17       58
16       51
15       51
18       33
20       30
19       26
21       24
22       17
23       16
24       15
27       11
29       10
25        8
28        8
26        8
31        7
30        7
35        5
32        4
33        4
34        3
42        2
40        2
43        2
56        1
39        1
41        1
37        1
Name: count, dtype: int64

previous_campaign_contacts:
previous_campai

In [36]:
economics = df[
    [
        "client_id",
        "cons_price_idx",
        "euribor_three_months"
    ]
].copy()

In [39]:
listOfEconomics= [
        "cons_price_idx",
        "euribor_three_months"
    ]

In [40]:
for col in listOfEconomics:
    print(f"\n{col}:")
    print(economics[col].value_counts(dropna=False))



cons_price_idx:
cons_price_idx
93.994    7763
93.918    6685
92.893    5794
93.444    5175
94.465    4374
93.200    3616
93.075    2458
92.201     770
92.963     715
92.431     447
92.649     357
94.215     311
94.199     303
92.843     282
92.379     267
93.369     264
94.027     233
94.055     229
93.876     212
94.601     204
92.469     178
93.749     174
92.713     172
94.767     128
93.798      67
92.756      10
Name: count, dtype: int64

euribor_three_months:
euribor_three_months
4.857    2868
4.962    2613
4.963    2487
4.961    1902
4.856    1210
         ... 
0.969       1
0.933       1
0.956       1
1.045       1
1.047       1
Name: count, Length: 316, dtype: int64
